# 01 — Data Ingestion and Cleaning

Load all raw datasets, clean and standardize them, convert coordinate systems to WGS84 (EPSG:4326), and save cleaned versions to `data/processed/`.

**Datasets loaded:**
1. Road network (Ministry of Transport)
2. Existing EV chargers (NAP — National Access Point)
3. Grid capacity (i-DE, Endesa, Viesgo)
4. EV registration forecast data (datos.gob.es parquets)
5. IMD traffic count stations (DGT)
6. Gas stations (MITECO)
7. Population (INE)
8. Tourism seasonality (INE)
9. Service areas (OSM/Hermes)
10. TEN-T corridors
11. Administrative boundaries (provinces, CCAA)

## Setup

In [7]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
# If running from notebooks/ folder, adjust path to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import geopandas as gpd
from pathlib import Path

from src.data_loading import (
    load_road_network, load_nap_charging_points, load_grid_capacity,
    load_ev_forecast, load_imd_traffic, load_gas_stations,
    load_population, load_tourism_seasonal, load_service_areas,
    load_tent_corridors, load_provinces, load_ccaa,
)

PROCESSED = Path('data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Setup complete. Project root:', os.getcwd())

Setup complete. Project root: /Users/nicolaswilches/ds/projects/iberdrola-ev-network


## 1. Road Network (Ministry of Transport)

In [2]:
roads = load_road_network()
print(f"Road network: {roads.shape[0]} segments, CRS: {roads.crs}")
print(f"Columns: {roads.columns.tolist()}")
print("\nRoad type distribution:")
print(roads['Tipo_de_via'].value_counts())

# Clean: drop segments without road name, validate geometry
roads = roads.dropna(subset=['Carretera'])
roads['geometry'] = roads.geometry.make_valid()

# Add useful columns
roads['length_km'] = roads['Longitud'] / 1000.0
roads['road_prefix'] = roads['Carretera'].str.extract(r'^(AP|A|N|[A-Z]+)')

# Save
roads.to_file(PROCESSED / 'roads_clean.geojson', driver='GeoJSON')
print(f"\nSaved: roads_clean.geojson ({roads.shape[0]} segments)")
roads.head()

Road network: 1602 segments, CRS: EPSG:4326
Columns: ['Carretera', 'Tipo_de_via', 'Titular', 'TENT', 'TENT_red_basica', 'TENT_corredor', 'PK_inicio', 'PK_fin', 'Longitud', 'Valido_desde', 'Valido_hasta', 'geometry']

Road type distribution:
Tipo_de_via
Carretera convencional     789
Autopista libre\Autovía    534
Multicarril                189
Autopista peaje             79
Name: count, dtype: int64

Saved: roads_clean.geojson (1602 segments)


,Carretera,Tipo_de_via,Titular,TENT,TENT_red_basica,TENT_corredor,PK_inicio,PK_fin,Longitud,Valido_desde,Valido_hasta,geometry,length_km,road_prefix
0,A-7S,Autopista peaje,Estado,SI,Comprehensive,NaN,1042.61,1046.860,5322.812012,1761523200000,None,"LINESTRING (-4.87023 36.51993, -4.87025 36.519...",5.322812,A
1,A-7S,Autopista libre\Autovía,Estado,SI,Comprehensive,NaN,1046.86,1047.220,450.830566,1761523200000,None,"LINESTRING (-4.92806 36.51281, -4.92815 36.512...",0.450831,A
2,A-7S,Autopista libre\Autovía,Estado,NO,NaN,NaN,1047.22,1068.550,22389.957031,1761523200000,None,"LINESTRING (-4.93283 36.51387, -4.93305 36.513...",22.389957,A
3,A-7S,Autopista peaje,Estado,SI,Comprehensive,NaN,1068.55,1070.830,2587.326416,1761523200000,None,"LINESTRING (-5.13842 36.43574, -5.13867 36.435...",2.587326,A
4,A-7S,Multicarril,Estado,SI,Comprehensive,NaN,1070.83,1091.065,20452.484375,1761523200000,None,"LINESTRING (-5.16676 36.43302, -5.16682 36.433...",20.452484,A


## 2. Existing EV Charging Points (NAP)

Parse the 83 MB DATEX II XML file using streaming `lxml.etree.iterparse`. This extracts ~12,000 charging sites with coordinates, connector count, and max power.

In [4]:
chargers = load_nap_charging_points()
print(f"NAP Charging Points: {chargers.shape[0]} sites")
print(f"Lat range: {chargers.latitude.min():.2f} – {chargers.latitude.max():.2f}")
print(f"Lon range: {chargers.longitude.min():.2f} – {chargers.longitude.max():.2f}")
print(f"Max power (kW): {chargers.max_power_kw.describe()}")
print("\nTop provinces:")
print(chargers['province'].value_counts().head(10))

# Validate coordinates are within Spain
chargers = chargers[
    (chargers.latitude >= 27) & (chargers.latitude <= 44) &
    (chargers.longitude >= -19) & (chargers.longitude <= 5)
]
chargers = chargers.drop_duplicates(subset=['site_id'])

# Save
chargers.to_csv(PROCESSED / 'chargers_clean.csv', index=False)
print(f"\nSaved: chargers_clean.csv ({chargers.shape[0]} sites)")
chargers.head()

NAP Charging Points: 12072 sites
Lat range: 27.75 – 43.69
Lon range: -18.02 – 4.29
Max power (kW): count    12072.000000
mean        49.884423
std         58.823147
min          0.060000
25%         22.000000
50%         22.170000
75%         50.000000
max       1000.000000
Name: max_power_kw, dtype: float64

Top provinces:
province
Barcelona            1500
Madrid               1398
Valencia/València     700
Alicante/Alacant      656
Balears, Illes        519
Sevilla               379
Asturias              348
Girona                343
Málaga                339
Murcia                331
Name: count, dtype: int64

Saved: chargers_clean.csv (12072 sites)


,site_id,name,latitude,longitude,address,municipality,province,postcode,n_connectors,max_power_kw
0,2023002088,Centro Porsche Baleares CP2,39.5948,2.635860,Camí dels Reis 166,Palma,"Balears, Illes",7011,1,350.0
1,2023002089,Centro Porsche Baleares CP1,39.5948,2.635900,Camí dels Reis 166,Palma,"Balears, Illes",7011,1,350.0
2,2023002112,Centro Porsche Alicante estación de carga 2,38.3466,-0.522950,Carrer Riu Turia 16,Alicante/Alacant,Alicante/Alacant,3006,1,350.0
3,2023002113,Centro Porsche Alicante estación de carga 1,38.3460,-0.523464,Carrer Riu Turia 16,Alicante/Alacant,Alicante/Alacant,3006,1,350.0
4,2023002090,Centro Porsche Asturias CP 2,43.3948,-5.816220,Pol. Industrial Los Peñones 18,Siero,Asturias,33420,1,320.0


## 3. Grid Capacity (i-DE, Endesa, Viesgo)

Load substation-level capacity data from three electricity distributors. Each has different file encoding, column names, and number formats. UTM coordinates (EPSG:25830) are converted to WGS84.

**Key caveat:** Endesa files are "generación" (generation capacity), not "demanda" — we use available capacity as a proxy for grid headroom. This is documented in assumptions.md.

In [5]:
grid = load_grid_capacity('all')
print(f"Grid Capacity: {grid.shape[0]} rows")
print("\nDistributor breakdown:")
print(grid['distributor_network'].value_counts())
print("\nCapacity stats (MW):")
print(grid['available_capacity_mw'].describe())
print("\nCoordinate ranges:")
print(f"  Lat: {grid.latitude.min():.2f} – {grid.latitude.max():.2f}")
print(f"  Lon: {grid.longitude.min():.2f} – {grid.longitude.max():.2f}")
print(f"\nProvinces covered: {grid.provincia.nunique()}")

# Save
grid.to_csv(PROCESSED / 'grid_capacity_unified.csv', index=False)
print(f"\nSaved: grid_capacity_unified.csv ({grid.shape[0]} rows)")
grid.head()

Grid Capacity: 4990 rows

Distributor breakdown:
distributor_network
i-DE      3016
Endesa    1797
Viesgo     177
Name: count, dtype: int64

Capacity stats (MW):
count    4990.000000
mean        5.083216
std        25.148220
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       475.410000
Name: available_capacity_mw, dtype: float64

Coordinate ranges:
  Lat: 36.03 – 43.59
  Lon: -7.89 – 3.15

Provinces covered: 47

Saved: grid_capacity_unified.csv (4990 rows)


,substation_name,provincia,municipio,utm_x,utm_y,available_capacity_mw,voltage_kv,distributor_network,latitude,longitude
0,3102,Araba/Álava,Ayala/Aiara,499100.028810,4.770106e+06,0.0,30.0,i-DE,43.083669,-3.011056
1,3017,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,0.0,13.2,i-DE,42.856076,-2.707190
2,3017,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,0.0,30.0,i-DE,42.856076,-2.707190
3,3017,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,0.0,13.2,i-DE,42.856076,-2.707190
4,3017,Araba/Álava,Vitoria-Gasteiz,523922.106338,4.744874e+06,0.0,30.0,i-DE,42.856076,-2.707190


## 4. EV Registration Forecast Data (datos.gob.es)

Load 108 monthly parquet files (2015–2023) of vehicle registration microdata. Filter to new BEV+PHEV registrations and aggregate to a monthly time series. This feeds the SARIMA model in Notebook 02.

In [9]:
ev_ts = load_ev_forecast()
print(f"EV Registration Time Series: {ev_ts.shape[0]} months")
print(f"Date range: {ev_ts.date.min()} – {ev_ts.date.max()}")
print(f"Total BEV+PHEV registrations (2015–2023): {ev_ts.ev_registrations.sum():,}")
print("\nMonthly stats:")
print(ev_ts['ev_registrations'].describe())

# Save
ev_ts.to_csv(PROCESSED / 'ev_monthly_registrations.csv', index=False)
print(f"\nSaved: ev_monthly_registrations.csv ({ev_ts.shape[0]} rows)")
ev_ts.tail(12)

EV Registration Time Series: 108 months
Date range: 2015-01-01 00:00:00 – 2023-12-01 00:00:00
Total BEV+PHEV registrations (2015–2023): 457,153

Monthly stats:
count      108.000000
mean      4232.898148
std       4122.175659
min        109.000000
25%        659.750000
50%       2601.500000
75%       7652.000000
max      14701.000000
Name: ev_registrations, dtype: float64

Saved: ev_monthly_registrations.csv (108 rows)


,year_month,ev_registrations,date
96,2023_01,8999,2023-01-01
97,2023_02,10155,2023-02-01
98,2023_03,12803,2023-03-01
99,2023_04,9892,2023-04-01
100,2023_05,13370,2023-05-01
101,2023_06,14463,2023-06-01
102,2023_07,11121,2023-07-01
103,2023_08,9043,2023-08-01
104,2023_09,10385,2023-09-01
105,2023_10,13806,2023-10-01


## 5. IMD Traffic Count Stations (DGT)

In [10]:
imd = load_imd_traffic()
print(f"IMD Traffic Stations: {imd.shape[0]} stations, CRS: {imd.crs}")
print(f"Columns: {imd.columns.tolist()}")
print("\nIMD total stats:")
if 'IMD_total' in imd.columns:
    print(imd['IMD_total'].describe())

# Save
imd.to_file(PROCESSED / 'imd_traffic_clean.geojson', driver='GeoJSON')
print(f"\nSaved: imd_traffic_clean.geojson ({imd.shape[0]} stations)")
imd.head()

IMD Traffic Stations: 14066 stations, CRS: EPSG:4326
Columns: ['Carretera', 'PK', 'Estacion', 'Via_servicio', 'Tipo_aforo', 'IMD_total', 'IMD_ligeros', 'IMD_pesados', 'IMD_asc', 'IMD_desc', 'Carriles', 'Vel_ligeros', 'Vel_pesados', 'Vel_global', 'Titular', 'TENT', 'TENT_red_basica', 'TENT_corredor', 'Valido_desde', 'Valido_hasta', 'geometry']

IMD total stats:
count     14066.000000
mean      16154.268804
std       22969.741617
min           0.000000
25%        2752.000000
50%        8155.000000
75%       19199.500000
max      190304.000000
Name: IMD_total, dtype: float64

Saved: imd_traffic_clean.geojson (14066 stations)


,Carretera,PK,Estacion,Via_servicio,Tipo_aforo,IMD_total,IMD_ligeros,IMD_pesados,IMD_asc,IMD_desc,...,Vel_ligeros,Vel_pesados,Vel_global,Titular,TENT,TENT_red_basica,TENT_corredor,Valido_desde,Valido_hasta,geometry
0,A-7,533.125,A-233-2,NO,Secundaria,37234,29683,7551,18485.0,18749.0,...,NaN,NaN,NaN,Estado,SI,Core,Mediterráneo,1642118400000,None,POINT (-0.84295 38.19174)
1,A-7,543.700,A-184-2,NO,Secundaria,33292,25894,7398,16531.0,16761.0,...,NaN,NaN,NaN,Estado,SI,Core,Mediterráneo,1642118400000,None,POINT (-0.93844 38.14016)
2,A-70,2.440,A-208-2,NO,Secundaria,46281,43607,2674,22960.0,23321.0,...,NaN,NaN,NaN,Estado,SI,Core,Mediterráneo,1642118400000,None,POINT (-0.39757 38.44138)
3,A-70,7.200,A-209-2,NO,Secundaria,61683,57936,3747,26509.0,35174.0,...,NaN,NaN,NaN,Estado,SI,Core,Mediterráneo,1642118400000,None,POINT (-0.43717 38.41111)
4,A-70,13.090,A-182-2,NO,Secundaria,41327,38277,3050,20453.0,20874.0,...,NaN,NaN,NaN,Estado,SI,Core,Mediterráneo,1642118400000,None,POINT (-0.48093 38.38473)


## 6. Supplementary Datasets

Gas stations, population, tourism, service areas, TEN-T corridors, and administrative boundaries.

In [11]:
# Gas stations (MITECO) — potential candidate locations for co-location
gas = load_gas_stations()
print(f"Gas Stations: {gas.shape[0]} stations")
gas.to_csv(PROCESSED / 'gas_stations_clean.csv', index=False)
print("Saved: gas_stations_clean.csv")

# Population (INE)
pop = load_population()
print(f"\nPopulation: {pop.shape[0]} municipalities")
pop.to_csv(PROCESSED / 'population_municipal.csv', index=False)
print("Saved: population_municipal.csv")

# Tourism seasonality (INE)
tourism = load_tourism_seasonal()
print(f"\nTourism: {tourism.shape[0]} rows")
tourism.to_csv(PROCESSED / 'tourism_seasonal.csv', index=False)
print("Saved: tourism_seasonal.csv")

# Service areas (OSM/Hermes)
svc = load_service_areas()
print(f"\nService Areas: {svc.shape[0]} areas")
svc.to_file(PROCESSED / 'service_areas_clean.geojson', driver='GeoJSON')
print(f"Saved: service_areas_clean.geojson")

# TEN-T corridors
tent = load_tent_corridors()
print(f"\nTEN-T Corridors: {tent.shape[0]} segments")
tent.to_file(PROCESSED / 'tent_corridors.geojson', driver='GeoJSON')
print(f"Saved: tent_corridors.geojson")

# Province and CCAA boundaries
provinces = load_provinces()
provinces.to_file(PROCESSED / 'provinces.geojson', driver='GeoJSON')
print(f"\nProvinces: {provinces.shape[0]} → saved")

ccaa = load_ccaa()
ccaa.to_file(PROCESSED / 'ccaa.geojson', driver='GeoJSON')
print(f"CCAA: {ccaa.shape[0]} → saved")

Gas Stations: 12216 stations
Saved: gas_stations_clean.csv

Population: 8132 municipalities
Saved: population_municipal.csv

Tourism: 6760 rows
Saved: tourism_seasonal.csv

Service Areas: 113 areas
Saved: service_areas_clean.geojson

TEN-T Corridors: 77 segments
Saved: tent_corridors.geojson

Provinces: 52 → saved
CCAA: 19 → saved


## Summary — All Processed Files

In [12]:
import os

summary = []
for f in sorted(PROCESSED.glob('*')):
    size_kb = os.path.getsize(f) / 1024
    summary.append({'file': f.name, 'size_kb': round(size_kb, 1)})

summary_df = pd.DataFrame(summary)
print(f"Total processed files: {len(summary_df)}")
print(f"Total size: {summary_df['size_kb'].sum() / 1024:.1f} MB")
summary_df

Total processed files: 13
Total size: 91.8 MB


,file,size_kb
0,ccaa.geojson,728.0
1,chargers_clean.csv,1424.1
2,ev_monthly_registrations.csv,2.5
3,ev_projection_2027.csv,4.7
4,gas_stations_clean.csv,1518.0
5,grid_capacity_unified.csv,520.0
6,imd_traffic_clean.geojson,7708.0
7,population_municipal.csv,377.0
8,provinces.geojson,805.9
9,roads_clean.geojson,75255.4
